# q04：ベースラインと改良手法の比較および考察

基礎レベル4では，最終的なベースラインと改良手法を同一条件で比較する．

比較条件は，10 msフレーム，trainはA1からA6，testはA7からA9，評価指標はmacro-F1とする．

In [ ]:
import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import json



from pathlib import Path

ROOT = Path("./data")
INPUT_DIR = ROOT / "inputs"
ANNOTATION_DIR = ROOT / "annotations"
RESULT_DIR = ROOT / "results"

ANNOTATION_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_IDS = [f"A{i}" for i in range(1, 7)]
TEST_IDS = [f"A{i}" for i in range(7, 10)]
ALL_IDS = TRAIN_IDS + TEST_IDS

FRAME_SEC = 0.010
TARGET_FS = 16000

def wav_path(utt_id: str) -> Path:
    return INPUT_DIR / f"{utt_id}.wav"

def interval_path(utt_id: str) -> Path:
    return ANNOTATION_DIR / f"{utt_id}_intervals.csv"

def label_path(utt_id: str) -> Path:
    return ANNOTATION_DIR / f"{utt_id}_labels.csv"



import numpy as np
import pandas as pd
import soundfile as sf

EPS = 1e-12

def load_audio(path, target_fs: int = 16000):
    """WAVを読み込み，モノラルfloat32波形として返す．"""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"音声ファイルが見つかりません: {path}")
    x, fs = sf.read(path, dtype="float32")
    if x.ndim == 2:
        x = x.mean(axis=1)
    if fs != target_fs:
        raise ValueError(f"{path} のサンプリング周波数が {fs} Hz です．{target_fs} Hzで収録した音声を使用してください．")
    return x.astype(np.float32), fs

def frame_signal(x: np.ndarray, fs: int, frame_sec: float = 0.010):
    """10 msごとに非重複フレーム化する．不足分は0で埋める．"""
    frame_len = int(round(frame_sec * fs))
    n_frames = int(np.ceil(len(x) / frame_len))
    pad_len = n_frames * frame_len - len(x)
    if pad_len > 0:
        x = np.pad(x, (0, pad_len))
    frames = x.reshape(n_frames, frame_len)
    times = np.arange(n_frames) * frame_sec
    return frames, times

def hann_window(n: int):
    return np.hanning(n).astype(np.float32)

def short_time_log_energy(x: np.ndarray, fs: int, frame_sec: float = 0.010, use_hann: bool = True):
    frames, times = frame_signal(x, fs, frame_sec)
    if use_hann:
        frames = frames * hann_window(frames.shape[1])[None, :]
    energy = np.mean(frames ** 2, axis=1)
    log_energy = 10.0 * np.log10(EPS + energy)
    return log_energy, times

def relative_energy(log_energy: np.ndarray, percentile: float = 20.0):
    noise_floor = np.percentile(log_energy, percentile)
    score = log_energy - noise_floor
    return score, noise_floor

def apply_hangover(pred: np.ndarray, hangover_frames: int):
    """1から0へ切り替わった直後の指定フレーム数を1にする．"""
    pred = np.asarray(pred, dtype=np.int64).copy()
    if hangover_frames <= 0:
        return pred
    out = pred.copy()
    n = len(pred)
    for m in range(n - 1):
        if pred[m] == 1 and pred[m + 1] == 0:
            end = min(n, m + 1 + hangover_frames)
            out[m + 1:end] = 1
    return out

def labels_from_intervals(intervals, n_frames: int, frame_sec: float = 0.010):
    """[(start_sec, end_sec), ...]を10 msフレームラベルへ変換する．"""
    labels = np.zeros(n_frames, dtype=np.int64)
    for start_sec, end_sec in intervals:
        start = max(0, int(np.floor(start_sec / frame_sec)))
        end = min(n_frames, int(np.ceil(end_sec / frame_sec)))
        labels[start:end] = 1
    return labels

def save_labels_from_intervals(utt_id: str, intervals, duration_sec: float):
    """区間アノテーションCSVとフレームラベルCSVを保存する．"""
    n_frames = int(np.ceil(duration_sec / FRAME_SEC))
    interval_df = pd.DataFrame(intervals, columns=["start_sec", "end_sec"])
    interval_df["label"] = 1
    interval_df.to_csv(interval_path(utt_id), index=False)

    labels = labels_from_intervals(intervals, n_frames, FRAME_SEC)
    label_df = pd.DataFrame({
        "frame": np.arange(n_frames),
        "start_sec": np.arange(n_frames) * FRAME_SEC,
        "end_sec": (np.arange(n_frames) + 1) * FRAME_SEC,
        "label": labels
    })
    label_df.to_csv(label_path(utt_id), index=False)
    return interval_df, label_df

def load_frame_labels(utt_id: str, n_frames: int):
    """q01で保存したフレームラベルを読み込む．長さが合わない場合は切り詰めまたは0埋めする．"""
    path = label_path(utt_id)
    if not path.exists():
        raise FileNotFoundError(f"アノテーションが見つかりません: {path}")
    labels = pd.read_csv(path)["label"].to_numpy(dtype=np.int64)
    if len(labels) < n_frames:
        labels = np.pad(labels, (0, n_frames - len(labels)))
    elif len(labels) > n_frames:
        labels = labels[:n_frames]
    return labels

def confusion_binary(y_true: np.ndarray, y_pred: np.ndarray, positive_label: int = 1):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    tp = int(np.sum((y_true == positive_label) & (y_pred == positive_label)))
    fp = int(np.sum((y_true != positive_label) & (y_pred == positive_label)))
    fn = int(np.sum((y_true == positive_label) & (y_pred != positive_label)))
    tn = int(np.sum((y_true != positive_label) & (y_pred != positive_label)))
    return {"TP": tp, "FP": fp, "FN": fn, "TN": tn}

def f1_from_counts(tp: int, fp: int, fn: int):
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return f1, precision, recall

def macro_f1_score(y_true: np.ndarray, y_pred: np.ndarray):
    """音声クラスと非音声クラスのF1平均を返す．"""
    c_s = confusion_binary(y_true, y_pred, positive_label=1)
    f1_s, p_s, r_s = f1_from_counts(c_s["TP"], c_s["FP"], c_s["FN"])

    c_n = confusion_binary(1 - y_true, 1 - y_pred, positive_label=1)
    f1_n, p_n, r_n = f1_from_counts(c_n["TP"], c_n["FP"], c_n["FN"])

    return {
        "macro_f1": (f1_s + f1_n) / 2,
        "f1_speech": f1_s,
        "precision_speech": p_s,
        "recall_speech": r_s,
        "f1_non_speech": f1_n,
        "precision_non_speech": p_n,
        "recall_non_speech": r_n,
        **c_s
    }

def load_dataset(utt_ids):
    """音声，エネルギー，正解ラベルをIDごとに読み込む．"""
    data = {}
    for utt_id in utt_ids:
        x, fs = load_audio(wav_path(utt_id), TARGET_FS)
        E, times = short_time_log_energy(x, fs, FRAME_SEC, use_hann=True)
        y = load_frame_labels(utt_id, len(E))
        data[utt_id] = {"x": x, "fs": fs, "E": E, "times": times, "y": y}
    return data

def concat_labels_and_preds(y_list, pred_list):
    return np.concatenate(y_list), np.concatenate(pred_list)

## 1．q02とq03で保存した最良パラメータの読み込み

In [ ]:
with open(RESULT_DIR / "baseline_best_params.json", "r", encoding="utf-8") as f:
    best_baseline = json.load(f)

with open(RESULT_DIR / "improved_best_params.json", "r", encoding="utf-8") as f:
    best_improved = json.load(f)

print("best baseline")
print(json.dumps(best_baseline, ensure_ascii=False, indent=2))

print("best improved")
print(json.dumps(best_improved, ensure_ascii=False, indent=2))

## 2．ベースラインと改良手法の関数

In [ ]:
def predict_energy_only(E: np.ndarray, theta: float):
    return (E >= theta).astype(np.int64)

def predict_relative_energy(E: np.ndarray, theta: float):
    S, noise_floor = relative_energy(E, percentile=20.0)
    return (S >= theta).astype(np.int64)

def predict_relative_energy_hangover(E: np.ndarray, theta: float, Lh: int):
    pred0 = predict_relative_energy(E, theta)
    return apply_hangover(pred0, Lh)

def predict_best_baseline_from_audio(x: np.ndarray, fs: int):
    E, times = short_time_log_energy(x, fs, FRAME_SEC, use_hann=True)
    method = best_baseline["method"]
    theta = float(best_baseline["theta"])
    Lh = int(best_baseline.get("Lh", 0))

    if method == "energy_only":
        pred = predict_energy_only(E, theta)
    elif method == "relative_energy":
        pred = predict_relative_energy(E, theta)
    elif method == "relative_energy_hangover":
        pred = predict_relative_energy_hangover(E, theta, Lh)
    else:
        raise ValueError(f"unknown baseline method: {method}")

    return pred, times

def speech_band_log_energy(x: np.ndarray, fs: int, frame_sec: float = 0.010,
                           low_hz: float = 300.0, high_hz: float = 3400.0,
                           use_hann: bool = True):
    frames, times = frame_signal(x, fs, frame_sec)
    if use_hann:
        frames = frames * hann_window(frames.shape[1])[None, :]

    spec = np.fft.rfft(frames, axis=1)
    freqs = np.fft.rfftfreq(frames.shape[1], d=1.0 / fs)
    band = (freqs >= low_hz) & (freqs <= high_hz)

    power = np.mean(np.abs(spec[:, band]) ** 2, axis=1)
    band_energy = 10.0 * np.log10(EPS + power)
    return band_energy, times

def zero_crossing_rate(x: np.ndarray, fs: int, frame_sec: float = 0.010):
    frames, times = frame_signal(x, fs, frame_sec)
    signs = np.signbit(frames)
    zcr = np.mean(signs[:, 1:] != signs[:, :-1], axis=1)
    return zcr, times

def median_filter_binary(pred: np.ndarray, width: int):
    pred = np.asarray(pred, dtype=np.int64)
    if width <= 1:
        return pred.copy()
    pad = width // 2
    padded = np.pad(pred, (pad, pad), mode="edge")
    out = np.zeros_like(pred)
    for i in range(len(pred)):
        out[i] = 1 if np.median(padded[i:i + width]) >= 0.5 else 0
    return out

def remove_short_speech_segments(pred: np.ndarray, min_speech_frames: int):
    pred = np.asarray(pred, dtype=np.int64).copy()
    if min_speech_frames <= 1:
        return pred
    n = len(pred)
    i = 0
    while i < n:
        if pred[i] == 0:
            i += 1
            continue
        start = i
        while i < n and pred[i] == 1:
            i += 1
        end = i
        if end - start < min_speech_frames:
            pred[start:end] = 0
    return pred

def predict_best_improved_from_audio(x: np.ndarray, fs: int):
    band_E, times = speech_band_log_energy(x, fs, FRAME_SEC)
    S_band, _ = relative_energy(band_E, percentile=20.0)
    zcr, _ = zero_crossing_rate(x, fs, FRAME_SEC)

    pred = (S_band >= float(best_improved["theta"])).astype(np.int64)
    pred = pred & (zcr >= float(best_improved["zcr_min"])) & (zcr <= float(best_improved["zcr_max"]))
    pred = pred.astype(np.int64)
    pred = median_filter_binary(pred, int(best_improved["median_width"]))
    pred = remove_short_speech_segments(pred, int(best_improved["min_speech_frames"]))
    pred = apply_hangover(pred, int(best_improved["Lh"]))

    return pred, times, S_band, zcr

## 3．評価関数

In [ ]:
def evaluate_method_on_ids(utt_ids, method_name: str):
    y_all = []
    pred_all = []
    rows = []

    for uid in utt_ids:
        x, fs = load_audio(wav_path(uid), TARGET_FS)

        if method_name == "baseline":
            pred, times = predict_best_baseline_from_audio(x, fs)
        elif method_name == "improved":
            pred, times, S_band, zcr = predict_best_improved_from_audio(x, fs)
        else:
            raise ValueError(method_name)

        y = load_frame_labels(uid, len(pred))
        metrics = macro_f1_score(y, pred)
        rows.append({"utt_id": uid, "method": method_name, **metrics})

        y_all.append(y)
        pred_all.append(pred)

    y_all, pred_all = concat_labels_and_preds(y_all, pred_all)
    total_metrics = macro_f1_score(y_all, pred_all)
    return {"method": method_name, **total_metrics}, pd.DataFrame(rows)

## 4．trainとtestでの比較

In [ ]:
summary_rows = []
per_file_rows = []

for split_name, ids in [("train", TRAIN_IDS), ("test", TEST_IDS)]:
    for method_name in ["baseline", "improved"]:
        summary, per_file = evaluate_method_on_ids(ids, method_name)
        summary["split"] = split_name
        per_file["split"] = split_name

        summary_rows.append(summary)
        per_file_rows.append(per_file)

summary_df = pd.DataFrame(summary_rows)
per_file_df = pd.concat(per_file_rows, ignore_index=True)

summary_df.to_csv(RESULT_DIR / "comparison_summary.csv", index=False)
per_file_df.to_csv(RESULT_DIR / "comparison_per_file.csv", index=False)

display(summary_df[["split", "method", "macro_f1", "f1_speech", "f1_non_speech",
                    "precision_speech", "recall_speech", "TP", "FP", "FN", "TN"]])
display(per_file_df[["split", "utt_id", "method", "macro_f1", "f1_speech", "f1_non_speech", "TP", "FP", "FN", "TN"]])

## 5．結果の可視化

In [ ]:
labels = []
scores = []
for _, row in summary_df.iterrows():
    labels.append(f'{row["split"]}\n{row["method"]}')
    scores.append(row["macro_f1"])

plt.figure(figsize=(8, 4))
plt.bar(labels, scores)
plt.ylim(0, 1)
plt.ylabel("macro-F1")
plt.title("Baseline vs Improved VAD")
plt.grid(axis="y")
plt.show()

## 6．考察

以下のセルを実行すると，結果に基づく考察文の下書きを表示する．

In [ ]:
train_base = summary_df[(summary_df["split"] == "train") & (summary_df["method"] == "baseline")]["macro_f1"].iloc[0]
train_imp = summary_df[(summary_df["split"] == "train") & (summary_df["method"] == "improved")]["macro_f1"].iloc[0]
test_base = summary_df[(summary_df["split"] == "test") & (summary_df["method"] == "baseline")]["macro_f1"].iloc[0]
test_imp = summary_df[(summary_df["split"] == "test") & (summary_df["method"] == "improved")]["macro_f1"].iloc[0]

print("考察文の下書き")
print()
print(f"trainにおけるベースラインのmacro-F1は{train_base:.3f}，改良手法のmacro-F1は{train_imp:.3f}であった．")
print(f"testにおけるベースラインのmacro-F1は{test_base:.3f}，改良手法のmacro-F1は{test_imp:.3f}であった．")

if test_imp > test_base:
    print("testでも改良手法のmacro-F1が向上したため，音声帯域エネルギーやZCRを用いた特徴量設計が有効であったと考えられる．")
elif test_imp < test_base:
    print("testでは改良手法のmacro-F1が低下したため，trainに対してパラメータが過適合した可能性がある．")
else:
    print("testでは両手法のmacro-F1が同程度であり，今回の改良による明確な性能差は確認できなかった．")

print("FPが多い場合は非音声を音声と誤検出しており，しきい値を高くする，ZCR上限を厳しくする，短区間除去を強くする改善が考えられる．")
print("FNが多い場合は音声を見逃しており，しきい値を低くする，hangoverを長くする，音声帯域の範囲を見直す改善が考えられる．")